In [1]:
%load_ext autoreload
%autoreload 2

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

In [3]:
names = [
    'PowerCons',
    # 'RefrigerationDevices',
    # 'ScreenType',
    'Beef',
    'Coffee',
    'Ham',
    'ItalyPowerDemand',
    'Mallat',
    'Meat',
    'OliveOil',
    'Strawberry',
    'Wine'
]

In [ ]:
import numpy as np
from stock_embeddings.ucr import (
    evaluate_model_sklearn,
    get_kNN_accuracy_MF_UCR,
    evaluate_resampling_UCR,
)
from stock_embeddings.models.matrix_factorization import MatrixFactorization

In [5]:
from warnings import simplefilter
from sklearn.exceptions import ConvergenceWarning, UndefinedMetricWarning
simplefilter("ignore", category=ConvergenceWarning)
simplefilter("ignore", category=UndefinedMetricWarning)

## Learn and save embeddings for each dataset

In [ ]:
from stock_embeddings.ucr import UCR_Data
import time
from sklearn.svm import SVC
from tqdm import tqdm
import json

def train_embeddings(data, save=False, verbose=False):
    name = data.name
    if verbose:
        print(f"===== {name} =====")
        print(data.X.shape)
    similarity_setup = {
        "scaled":"normal", "return_similarity":True, "truncate":False
    }
    similarity_matrix = MatrixFactorization.get_euclidean_matrix(
        data.X, **similarity_setup
    )
    # similarity_matrix = MatrixFactorization.get_euclidean_matrix(
    #     np.diff(data.X, axis=1), scaled="normal", return_similarity=True, truncate=False
    # )
    start_time = time.time()
    train_setup = {
        "embedding_dim":32,
        "learning_rate":0.02,
        "epochs":300,
        "regularization_loss_weight":0.1,
        "pairwise_loss_weight":0.1,
        "noise_mask":False,
        "early_stopping":False,
    }
    model, losses, learning_rates = MatrixFactorization.train_MF_model(
        data.X.shape[0],
        similarity_matrix,
        verbose=True,
        **train_setup
    )
    embed_train_time = time.time() - start_time
    # Prepare data for evaluation
    embeddings = model.embeddings.weight.detach().numpy()

    if save:
        save_id = f"{train_setup['embedding_dim']}d"
        np.save(f"../embeddings/ucr/{name}_{save_id}.npy", embeddings)
        with open(f"../embeddings/ucr/{name}_setup_{save_id}.json", 'w', encoding='utf-8') as f:
            json.dump(similarity_setup | train_setup, f, ensure_ascii=False, indent=4)

    return model


In [ ]:
SAVE = False
save_id = "32d"

# names = sorted(list(meta_df[meta_df["learned_w"] == 0].Name))
verbose = True
results_dict = {}
for name in tqdm(names):
    if name in ["Crop"]:
        continue
    elif name in results_dict:
        continue
    try:
        data = UCR_Data(name)
        model = train_embeddings(data, save=False, verbose=False)
    except Exception as e:
        raise e

## Classification Evaluation

In [8]:
# Stops kernal crashing
import os
default_n_threads = 4
os.environ['OPENBLAS_NUM_THREADS'] = f"{default_n_threads}"
# os.environ['MKL_NUM_THREADS'] = f"{default_n_threads}"
# os.environ['OMP_NUM_THREADS'] = f"{default_n_threads}"

In [ ]:
from stock_embeddings.ucr import get_eval_df, add_meta_info, initialize_eval_df
from stock_embeddings.ucr import UCR_Data
import numpy as np
from tqdm import tqdm
from stock_embeddings.ucr import highlight_max_in_dataset
from collections import defaultdict
import time

embed_train_times = defaultdict(list)
for i in range(2):
    df = initialize_eval_df()
    # df = pl.read_parquet("eval_square_reg_2.parquet")
    for name in tqdm(names):
    # for name in ["ItalyPowerDemand", "PowerCons", "Mallat"]:
        if name == "Mallat":
            continue
        print(name)
        data = UCR_Data(name)
        # embeddings = np.load(f"../embeddings/ucr/{name}_32d.npy")
        start_time = time.time()
        model = train_embeddings(data, save=False, verbose=False, )
        embed_train_times[name].append(time.time()-start_time)
        embeddings = model.embeddings.weight.detach().numpy()
        save_string = f"eval_{i}.parquet"
        df = get_eval_df(data, embeddings, df=df, n_resamples=25, verbose=False,
                         scale=True, over_sampling=True, suffix="_scaledOS")
        df.write_parquet(save_string)
        # df = get_eval_df(data, embeddings, df=df, n_resamples=25, verbose=False,
        #                  scale=False, over_sampling=True, suffix="_noscaleOS")
        # df.write_parquet(save_string)
        df = get_eval_df(data, embeddings, df=df, n_resamples=25, verbose=False,
                         scale=True, over_sampling=False, suffix="_scaledNOS")
        df.write_parquet(save_string)
        # df = get_eval_df(data, embeddings, df=df, n_resamples=25, verbose=False,
        #                  scale=False, over_sampling=False, suffix="_noscaleNOS")
        # df.write_parquet(save_string)


##### Get latex table with max in bold

In [ ]:
from stock_embeddings.ucr import highlight_max_in_dataset, to_latex

# `df` is the live polars result frame from the loop above (the old eval_2.pkl
# is a local-only artifact that is not shipped).
highlighted = highlight_max_in_dataset(df)
print(to_latex(highlighted))
highlighted

## Clustering Evaluation

In [ ]:
from sklearn.metrics import adjusted_rand_score, adjusted_mutual_info_score
from sklearn.metrics import homogeneity_score, completeness_score, v_measure_score
from sklearn.metrics import fowlkes_mallows_score
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
from sklearn.cluster import KMeans

def clustering_evaluation(features, labels, random_state=42):
    n_clusters = len(set(labels))
    kmeans = KMeans(n_clusters=n_clusters, random_state=random_state)
    cluster_labels = kmeans.fit_predict(features)

    silhouette_avg = silhouette_score(features, cluster_labels)
    calinski_harabasz = calinski_harabasz_score(features, cluster_labels)
    davies_bouldin = davies_bouldin_score(features, cluster_labels)
    ari = adjusted_rand_score(labels, cluster_labels)
    ami = adjusted_mutual_info_score(labels, cluster_labels)
    homogeneity = homogeneity_score(labels, cluster_labels)
    completeness = completeness_score(labels, cluster_labels)
    v_measure = v_measure_score(labels, cluster_labels)
    fowlkes_mallows = fowlkes_mallows_score(labels, cluster_labels)

    return {
        'silhouette_score': silhouette_avg,
        'calinski_harabasz_score': calinski_harabasz,
        'davies_bouldin_score': davies_bouldin,
        'adjusted_rand_score': ari,
        'adjusted_mutual_info_score': ami,
        'homogeneity_score': homogeneity,
        'completeness_score': completeness,
        'v_measure_score': v_measure,
        'fowlkes_mallows_score': fowlkes_mallows
    }

In [ ]:
MatrixFactorization.plot_with_dimensionality_reduction(embeddings, data.y, method='pca')

In [ ]:
# Reuse the embeddings/data trained in the loop above (the Mallat_32d.npy
# artifact is local-only and not shipped).
cluster_out = clustering_evaluation(features=embeddings, labels=data.y)
{k: round(v, 4) for k, v in cluster_out.items()}

In [ ]:
c22_train, c22_test = data.get_sktime_features_unsupervised(
    X_train=data.X_train, X_test=data.X_test
)
c22_features = np.concatenate([c22_train, c22_test], axis=0)
cluster_out = clustering_evaluation(features=c22_features, labels=data.y)
{k: round(v, 4) for k, v in cluster_out.items()}